# 141 — Prompt Caching

**What you'll learn:**
- How caching activates: the ≥1024 token minimum and ephemeral TTL (~5 minutes of inactivity)
- The two API shapes: `system` as a plain string vs. `system` as a list with `cache_control`
- Pricing: regular input / cache write / cache read for Haiku and Sonnet
- Where caching helps: multi-turn chat, RAG with fixed context, repeated tool definitions
- How to measure tokens saved and latency reduction with `compute_savings()`
- Part A (no key required) vs. Part B (ANTHROPIC_API_KEY required)

---

**Source:** `examples/141-prompt-caching/` | **Model:** `claude-haiku-4-5-20251001` | **Key:** `ANTHROPIC_API_KEY`

In [ ]:
%pip install -q anthropic python-dotenv langgraph

---
## Part A — CPU-Safe Concept Demonstrations

No API key required. All cells in Part A work offline and are safe to run anywhere.

### Concept — How Prompt Caching Works

Anthropic's prompt caching lets you mark a prefix of your request (typically a large system prompt)
so the model can reuse a pre-computed representation on follow-up calls.

**Mechanics:**
1. You add `"cache_control": {"type": "ephemeral"}` to the system block.
2. **First call (cache write):** Anthropic processes and stores the prefix. You pay slightly *more* than the regular input price (1.25× on Haiku).
3. **Subsequent calls (cache read):** Anthropic reads from the KV cache. You pay 10× *less* than regular input price.
4. **TTL:** Cache entries expire after roughly 5 minutes of inactivity.
5. **Minimum size:** The prefix must be ≥ 1024 tokens (roughly 4 000 characters) or caching is silently skipped.

**Pricing:**

| Model | Regular input | Cache write | Cache read |
|---|---|---|---|
| claude-haiku-4-5 | \$0.25/MTok | \$0.30/MTok | \$0.03/MTok |
| claude-sonnet-4-5 | \$3.00/MTok | \$3.75/MTok | \$0.30/MTok |

> The savings compound across many queries: a 1 500-token system prompt reused 100 times
> costs ~\$0.004 with caching vs ~\$0.038 without (Haiku pricing).

### Concept — The Two API Shapes

The only change between a regular and a cache-enabled call is the type of the `system` parameter.

**Regular call** — `system` is a plain string:
```python
client.messages.create(
    model="claude-haiku-4-5-20251001",
    system="You are an expert...",          # plain string
    messages=[{"role": "user", "content": question}],
)
```

**Cache-enabled call** — `system` is a list with `cache_control`:
```python
client.messages.create(
    model="claude-haiku-4-5-20251001",
    system=[
        {
            "type": "text",
            "text": "You are an expert...",
            "cache_control": {"type": "ephemeral"}   # only this changes
        }
    ],
    messages=[{"role": "user", "content": question}],
)
```

The response then includes `usage.cache_read_input_tokens` — the number of tokens
served from cache (not billed at the full input rate).

### Concept — Where Caching Helps

| Use case | Cache benefit | Why |
|---|---|---|
| Multi-turn chat | High | Same large system prompt is sent every turn |
| RAG with fixed context | High | Retrieved documents are identical across calls |
| Repeated tool definitions | Medium | Tool JSON schema is sent with each request |
| Single one-off query | None | Cache write cost exceeds any read savings |
| Frequently changing prompts | None | Cache miss on every call — only write cost applies |

**Rule of thumb:** caching breaks even after ~2 reads on Haiku and ~3 reads on Sonnet (write is 1.25× input, read is 0.1× input).

In [ ]:
# Part A — Build the large system prompt
# Must exceed 1024 tokens (~4000 characters) for caching to activate.

SYSTEM_PROMPT = (
    "You are an expert on the Python programming language with deep knowledge of: "
    "language specification and PEPs, standard library internals, "
    "CPython implementation details, performance optimization techniques, "
    "concurrency patterns (threading, asyncio, multiprocessing), "
    "packaging, testing, and deployment best practices. "
    "Answer questions accurately and concisely. "
    + ("This system prompt is intentionally long to exceed the 1024-token caching threshold. " * 25)
)

char_count = len(SYSTEM_PROMPT)
est_tokens = char_count // 4

print(f"System prompt length : {char_count:,} chars")
print(f"Estimated tokens     : ~{est_tokens:,} tokens")
print(f"Exceeds 1024-token threshold: {est_tokens >= 1024}")
print()

# Show what happens at exactly 1023 tokens
short_prompt = "You are a helpful assistant. " * 30  # ~210 tokens
short_tokens = len(short_prompt) // 4
print(f"Short prompt: {len(short_prompt)} chars (~{short_tokens} tokens)")
print(f"Would caching activate? {short_tokens >= 1024}")
print("  ↳ At 1023 tokens the cache_control flag is accepted but caching does NOT activate.")

In [ ]:
# Part A — The two call shapes as Python dicts (no API call)
import json

question = "What is the GIL and how does it affect multithreading?"

# Shape 1: regular — system as plain string
regular_call = {
    "model": "claude-haiku-4-5-20251001",
    "max_tokens": 256,
    "system": SYSTEM_PROMPT,
    "messages": [{"role": "user", "content": question}],
}

# Shape 2: cached — system as list block with cache_control
cached_call = {
    "model": "claude-haiku-4-5-20251001",
    "max_tokens": 256,
    "system": [
        {
            "type": "text",
            "text": SYSTEM_PROMPT,
            "cache_control": {"type": "ephemeral"},
        }
    ],
    "messages": [{"role": "user", "content": question}],
}

print("--- Regular call ---")
print(f"  type(system)  : {type(regular_call['system']).__name__}")
print(f"  Top-level keys: {list(regular_call.keys())}")

print()
print("--- Cached call ---")
print(f"  type(system)  : {type(cached_call['system']).__name__} with {len(cached_call['system'])} block(s)")
print(f"  Block keys    : {list(cached_call['system'][0].keys())}")
print(f"  cache_control : {json.dumps(cached_call['system'][0]['cache_control'])}")

print()
print("Only the 'system' field differs — everything else is identical.")

In [ ]:
# Part A — compute_savings() demo with mock data
# Uses the same function as src/tools.py to keep Part A self-contained.

def compute_savings(uncached: list[dict], cached: list[dict]) -> dict:
    """Compute token and cost savings between uncached and cached runs."""
    cost_per_mtok = 0.25  # Haiku input price $/MTok
    uncached_tok = sum(r["input_tokens"] for r in uncached)
    # Billed tokens = input_tokens minus the portion served from cache
    cached_tok = sum(r["input_tokens"] - r["cache_read"] for r in cached)
    saved_tok = uncached_tok - cached_tok
    return {
        "uncached_total_input_tokens": uncached_tok,
        "cached_total_billed_tokens": cached_tok,
        "tokens_saved": saved_tok,
        "estimated_savings_usd": round(saved_tok / 1_000_000 * cost_per_mtok, 6),
        "avg_latency_uncached_ms": round(sum(r["latency_ms"] for r in uncached) / len(uncached)),
        "avg_latency_cached_ms": round(sum(r["latency_ms"] for r in cached) / len(cached)),
    }


# 3 questions, system prompt ~1500 tokens each call
MOCK_UNCACHED = [
    {"input_tokens": 1520, "cache_read": 0, "latency_ms": 820},
    {"input_tokens": 1518, "cache_read": 0, "latency_ms": 790},
    {"input_tokens": 1522, "cache_read": 0, "latency_ms": 810},
]

# Same total tokens, but 1400 served from cache each call
MOCK_CACHED = [
    {"input_tokens": 1520, "cache_read": 1400, "latency_ms": 210},
    {"input_tokens": 1518, "cache_read": 1400, "latency_ms": 195},
    {"input_tokens": 1522, "cache_read": 1400, "latency_ms": 205},
]

savings = compute_savings(MOCK_UNCACHED, MOCK_CACHED)

print("=== Simulated Savings (mock data) ===")
for k, v in savings.items():
    print(f"  {k}: {v}")

print()
pct_tok = round(savings["tokens_saved"] / savings["uncached_total_input_tokens"] * 100, 1)
pct_lat = round(
    (savings["avg_latency_uncached_ms"] - savings["avg_latency_cached_ms"])
    / savings["avg_latency_uncached_ms"] * 100, 1
)
print(f"Token reduction  : {pct_tok}%")
print(f"Latency reduction: {pct_lat}%")

In [ ]:
# Part A — Token threshold demo
# What the API returns when the system prompt is below 1024 tokens.

SHORT_PROMPT = "You are a helpful assistant."  # ~7 tokens, far below threshold
short_char_count = len(SHORT_PROMPT)
short_token_estimate = short_char_count // 4

print("=== Short prompt scenario ===")
print(f"  Prompt      : '{SHORT_PROMPT}'")
print(f"  Characters  : {short_char_count}")
print(f"  Est. tokens : ~{short_token_estimate}")
print()

# Simulate what the usage block would look like — no cache_read_input_tokens
mock_usage_no_cache = {
    "input_tokens": short_token_estimate + 10,  # + question tokens
    "output_tokens": 45,
    # cache_read_input_tokens is absent — caching was not active
}

print("Simulated usage block (caching NOT active):")
print(f"  {mock_usage_no_cache}")
print()
print("WARNING: cache_control was accepted by the API but caching did NOT activate.")
print("         Minimum prompt size for caching: 1024 tokens (~4000 characters).")
print()

# Compare to the large prompt from cell 7
print("Contrast with the large SYSTEM_PROMPT from Cell 7:")
print(f"  Est. tokens : ~{len(SYSTEM_PROMPT) // 4}  ← above threshold, caching activates")

### Exercise 1 — Compute Savings at Sonnet Prices

The `compute_savings()` function above is hardcoded for Haiku (`$0.25/MTok` input, `$0.03/MTok` cache read).

**Your task:** write `compute_savings_sonnet()` that uses Sonnet pricing:
- Regular input: `$3.00/MTok`
- Cache read: `$0.30/MTok`

Run it against `MOCK_UNCACHED` and `MOCK_CACHED` from Cell 9 and compare the absolute dollar savings.

```python
def compute_savings_sonnet(uncached: list[dict], cached: list[dict]) -> dict:
    input_price = 3.00    # TODO: Sonnet input $/MTok
    cache_read_price = 0.30  # TODO: Sonnet cache read $/MTok
    ...
```

*Hint: the token counts are identical — only the dollar multiplier changes.*

In [ ]:
# Exercise 1 — Answer key

def compute_savings_sonnet(uncached: list[dict], cached: list[dict]) -> dict:
    """Same logic as compute_savings() but at Sonnet pricing."""
    # Sonnet charges 12× more for input and 10× more for cache reads vs Haiku
    input_price = 3.00        # Sonnet regular input $/MTok
    cache_read_price = 0.30   # Sonnet cache read $/MTok  (vs $0.03 for Haiku)

    uncached_tok = sum(r["input_tokens"] for r in uncached)
    cached_billed_tok = sum(r["input_tokens"] - r["cache_read"] for r in cached)
    saved_tok = uncached_tok - cached_billed_tok

    # Cost if every token was billed at full input price (no caching)
    cost_uncached = uncached_tok / 1_000_000 * input_price
    # Cost with caching: billed tokens at input price + cache_read tokens at read price
    total_cache_read = sum(r["cache_read"] for r in cached)
    cost_cached = cached_billed_tok / 1_000_000 * input_price + total_cache_read / 1_000_000 * cache_read_price

    return {
        "tokens_saved": saved_tok,
        "cost_uncached_usd": round(cost_uncached, 6),
        "cost_cached_usd": round(cost_cached, 6),
        "estimated_savings_usd": round(cost_uncached - cost_cached, 6),
        "avg_latency_uncached_ms": round(sum(r["latency_ms"] for r in uncached) / len(uncached)),
        "avg_latency_cached_ms": round(sum(r["latency_ms"] for r in cached) / len(cached)),
    }


sonnet_savings = compute_savings_sonnet(MOCK_UNCACHED, MOCK_CACHED)
haiku_savings  = compute_savings(MOCK_UNCACHED, MOCK_CACHED)

print("=== Sonnet vs. Haiku savings on same mock data ===")
print(f"  Haiku  estimated savings : ${haiku_savings['estimated_savings_usd']}")
print(f"  Sonnet estimated savings : ${sonnet_savings['estimated_savings_usd']}")
print()
ratio = round(sonnet_savings["estimated_savings_usd"] / haiku_savings["estimated_savings_usd"], 1)
print(f"  Sonnet savings are {ratio}× larger in absolute dollar terms.")

### Exercise 2 — Restructure a Conversation for Caching

A common antipattern is embedding the system prompt inline in every user message:

```python
# BAD: system prompt duplicated in every message body
messages = [
    {"role": "user", "content": SYSTEM_PROMPT + "\n\nQ: What is the GIL?"},
    {"role": "assistant", "content": "The GIL is..."},
    {"role": "user", "content": SYSTEM_PROMPT + "\n\nQ: Explain __repr__."},
]
```

**Your task:** restructure this so the cacheable prefix lives in the dedicated `system` field
with `cache_control`, and the `messages` list contains only the actual conversation turns.

```python
# TODO: move SYSTEM_PROMPT here with cache_control
system = ...

# TODO: keep only the user/assistant turns here
messages = ...
```

In [ ]:
# Exercise 2 — Answer key

# BEFORE: system prompt embedded in every message turn
messages_before = [
    {"role": "user",      "content": SYSTEM_PROMPT + "\n\nQ: What is the GIL?"},
    {"role": "assistant", "content": "The GIL is a mutex that protects Python objects..."},
    {"role": "user",      "content": SYSTEM_PROMPT + "\n\nQ: Explain __repr__."},
]

total_chars_before = sum(len(m["content"]) for m in messages_before)

print("=== BEFORE: system prompt duplicated in messages ===")
print(f"  Message count      : {len(messages_before)}")
print(f"  Total content chars: {total_chars_before:,}")
print(f"  System prompt sent : {messages_before.count} times (once per user turn)")

print()

# AFTER: system prompt in dedicated cached field; messages = turns only
system_after = [
    {
        "type": "text",
        "text": SYSTEM_PROMPT,
        "cache_control": {"type": "ephemeral"},
    }
]

messages_after = [
    {"role": "user",      "content": "Q: What is the GIL?"},
    {"role": "assistant", "content": "The GIL is a mutex that protects Python objects..."},
    {"role": "user",      "content": "Q: Explain __repr__."},
]

total_chars_after = sum(len(m["content"]) for m in messages_after)

print("=== AFTER: system prompt in cached system field ===")
print(f"  system field chars : {len(SYSTEM_PROMPT):,} (sent once, cached after first call)")
print(f"  Message count      : {len(messages_after)}")
print(f"  Total content chars: {total_chars_after:,}")
print()

reduction = round((1 - total_chars_after / total_chars_before) * 100, 1)
print(f"  Message payload reduction: {reduction}% (system prompt no longer in each turn)")

---
## Part B — Live Anthropic API Calls

**Requirements:**
- `ANTHROPIC_API_KEY` set in your `.env` file or shell environment
- Internet connection to reach `api.anthropic.com`

**What Part B does:**
1. Sends 3 questions **without** caching — measures tokens billed and latency.
2. Sends the same 3 questions **with** `cache_control` — first call writes the cache (slightly higher cost), subsequent calls read from it.
3. Prints a comparison table and computes savings.

> **Note:** the very first cached call writes the cache prefix (slightly higher latency and cost than regular input). Calls 2 and 3 demonstrate the read benefit. In production you amortize the write cost over many reads.

In [ ]:
# Part B — Setup: load key, fail fast if missing
import os
from dotenv import load_dotenv

load_dotenv()

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
if not ANTHROPIC_API_KEY:
    raise EnvironmentError(
        "ANTHROPIC_API_KEY is not set. "
        "Add it to your .env file or run: export ANTHROPIC_API_KEY=sk-ant-..."
    )
print("ANTHROPIC_API_KEY loaded.")

In [ ]:
# Part B — 3 questions uncached
# Uses call_no_cache() from src/tools.py
import sys
sys.path.insert(0, ".")

import anthropic
from src.tools import call_no_cache

# Same large system prompt as Part A
SYSTEM_PROMPT_LIVE = (
    "You are an expert on the Python programming language with deep knowledge of: "
    "language specification and PEPs, standard library internals, "
    "CPython implementation details, performance optimization techniques, "
    "concurrency patterns (threading, asyncio, multiprocessing), "
    "packaging, testing, and deployment best practices. "
    "Answer questions accurately and concisely. "
    + ("This system prompt is intentionally long to exceed the 1024-token caching threshold. " * 25)
)

QUESTIONS = [
    "What is the GIL and how does it affect multithreading?",
    "Explain the difference between __str__ and __repr__.",
    "How does Python's garbage collector handle circular references?",
]

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

uncached_results = []
for q in QUESTIONS:
    result = call_no_cache(SYSTEM_PROMPT_LIVE, q, client)
    uncached_results.append(result)

print("=== Uncached Calls ===")
print(f"{'Q#':<4} {'input_tokens':>13} {'latency_ms':>11}")
print("-" * 32)
for i, r in enumerate(uncached_results):
    print(f"Q{i+1:<3} {r['input_tokens']:>13} {r['latency_ms']:>11}")

In [ ]:
# Part B — Same 3 questions with caching
# Uses call_with_cache() from src/tools.py
from src.tools import call_with_cache

cached_results = []
for q in QUESTIONS:
    result = call_with_cache(SYSTEM_PROMPT_LIVE, q, client)
    cached_results.append(result)

print("=== Cached Calls ===")
print(f"{'Q#':<4} {'input_tokens':>13} {'cache_read':>11} {'latency_ms':>11}")
print("-" * 44)
for i, r in enumerate(cached_results):
    print(f"Q{i+1:<3} {r['input_tokens']:>13} {r['cache_read']:>11} {r['latency_ms']:>11}")

print()
print("Note: Q1 cache_read may be 0 (cache write turn); Q2 and Q3 should show cache hits.")

In [ ]:
# Part B — Comparison table and savings
from src.tools import compute_savings

print("=== Side-by-Side Comparison ===")
header = f"{'Q#':<4} {'unc_tok':>8} {'unc_ms':>7} | {'cac_billed':>10} {'cac_read':>9} {'cac_ms':>7}"
print(header)
print("-" * len(header))
for i, (unc, cac) in enumerate(zip(uncached_results, cached_results)):
    billed = cac["input_tokens"] - cac["cache_read"]
    print(
        f"Q{i+1:<3} {unc['input_tokens']:>8} {unc['latency_ms']:>7} | "
        f"{billed:>10} {cac['cache_read']:>9} {cac['latency_ms']:>7}"
    )

print()
s = compute_savings(uncached_results, cached_results)
print("=== Summary ===")
print(f"  Tokens saved           : {s['tokens_saved']}")
tok_pct = round(s['tokens_saved'] / s['uncached_total_input_tokens'] * 100, 1) if s['uncached_total_input_tokens'] else 0
print(f"  Token reduction        : {tok_pct}%")
print(f"  Estimated savings (USD): ${s['estimated_savings_usd']}")
print(f"  Avg latency uncached   : {s['avg_latency_uncached_ms']} ms")
print(f"  Avg latency cached     : {s['avg_latency_cached_ms']} ms")
if s['avg_latency_uncached_ms'] > 0:
    lat_pct = round(
        (s['avg_latency_uncached_ms'] - s['avg_latency_cached_ms'])
        / s['avg_latency_uncached_ms'] * 100, 1
    )
    print(f"  Latency improvement    : {lat_pct}%")

---
## What's Next

- **Anthropic caching docs:** https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching
- **Example 142 (semantic-caching):** cache LLM responses by embedding similarity — avoid redundant API calls for semantically equivalent queries
- **Tool definition caching:** the same `cache_control` pattern works for large tool JSON schemas passed in each request
- **Multi-turn caching:** mark a growing conversation history prefix for caching as turns accumulate
- **Paper — KV-cache foundations:** "Efficient Memory Management for Large Language Model Serving with PagedAttention" (Kwon et al., 2023) — the server-side mechanism prompt caching builds on

### Caching Pricing Deep-Dive

Prompt caching is not free — the first call **writes** the cache (slightly more expensive),
but subsequent calls **read** from cache at a deeply discounted rate.

| Token type | Haiku price / MTok | Sonnet price / MTok | Notes |
|---|---|---|---|
| Regular input | $0.25 | $3.00 | Baseline — no cache |
| Cache write | $0.30 | $3.75 | ~1.2× regular — one-time cost |
| Cache read | $0.03 | $0.30 | **10× cheaper** — paid on reuse |
| Output | $1.25 | $15.00 | Same regardless of caching |

**Break-even:** caching pays off after just **1–2 reuses** of a prefix that costs more than the cache-write surcharge.

**When caching helps most:**
- Multi-turn chat with a large system prompt (same prefix every turn)
- RAG: fixed document context + many user questions
- Tool definitions: large tool JSON repeated across calls
- Batch evaluation: same judge prompt applied to 100s of samples

In [ ]:
# Caching ROI calculator
def cache_roi(prefix_tokens: int, n_calls: int, model: str = 'haiku') -> dict:
    prices = {
        'haiku':  {'regular': 0.25, 'write': 0.30, 'read': 0.03},
        'sonnet': {'regular': 3.00, 'write': 3.75, 'read': 0.30},
    }
    p = prices[model]
    no_cache_cost = prefix_tokens / 1e6 * p['regular'] * n_calls
    with_cache = (prefix_tokens / 1e6 * p['write']  # first call writes
                  + prefix_tokens / 1e6 * p['read'] * (n_calls - 1))  # rest read
    return {
        'model': model, 'prefix_tokens': prefix_tokens, 'n_calls': n_calls,
        'no_cache_usd': round(no_cache_cost, 6),
        'with_cache_usd': round(with_cache, 6),
        'savings_usd': round(no_cache_cost - with_cache, 6),
        'savings_pct': round((no_cache_cost - with_cache) / no_cache_cost * 100, 1),
    }

# Scenarios
for scenario in [
    (1500, 3, 'haiku'),    # small system prompt, few calls
    (5000, 10, 'haiku'),   # medium prompt, 10 questions
    (20000, 50, 'sonnet'), # large doc context, 50 questions
]:
    r = cache_roi(*scenario)
    print(f"{r['model']:7} | {r['prefix_tokens']:>6} tok | {r['n_calls']:>3} calls | "
          f"no-cache ${r['no_cache_usd']:>8.4f} | cached ${r['with_cache_usd']:>8.4f} | "
          f"saves {r['savings_pct']:>5.1f}%")

### When Caching Does NOT Activate

| Condition | Result | Why |
|---|---|---|
| Prefix < 1024 tokens | No cache | Minimum threshold not met |
| Prefix changes between calls | Cache miss | Cache key is the full prefix hash |
| > 5 min inactivity | Cache expiry | Ephemeral TTL reached |
| Different model version | Cache miss | Model is part of the cache key |
| System prompt modified | Cache miss | Even 1 character change = different prefix |

> **Key insight:** The cached prefix must be **identical** across calls.
> Construct your system prompt once, store it as a constant, and never modify it per-call.
> Per-call data (user query, context) goes in the `messages` array, not the system prompt.

### Exercise 1 — Sonnet Savings Calculator

The `compute_savings()` function in `src/tools.py` uses Haiku pricing.
Modify it to accept a `model` parameter and apply the correct prices for Sonnet too.

In [ ]:
# Exercise 1: Add a `model` parameter to compute_savings() and support 'sonnet' pricing.
# Sonnet: regular=$3.00/MTok, cache_read=$0.30/MTok
#
# TODO: implement compute_savings_v2(uncached, cached, model='haiku')
# and call it for both models with MOCK_UNCACHED / MOCK_CACHED from earlier.


In [ ]:
# Exercise 1 — Answer Key
MOCK_UNCACHED = [
    {'input_tokens': 1520, 'cache_read': 0, 'latency_ms': 820},
    {'input_tokens': 1518, 'cache_read': 0, 'latency_ms': 790},
    {'input_tokens': 1522, 'cache_read': 0, 'latency_ms': 810},
]
MOCK_CACHED = [
    {'input_tokens': 1520, 'cache_read': 1400, 'latency_ms': 210},
    {'input_tokens': 1518, 'cache_read': 1400, 'latency_ms': 195},
    {'input_tokens': 1522, 'cache_read': 1400, 'latency_ms': 205},
]

def compute_savings_v2(uncached, cached, model='haiku'):
    # The only difference is the cost per million tokens for regular input.
    # Cache read is already baked in via the cache_read field — we just
    # subtract it from the billed input tokens to get the effective cost.
    cost = {'haiku': 0.25, 'sonnet': 3.00}
    per_mtok = cost[model]
    uncached_tok = sum(r['input_tokens'] for r in uncached)
    cached_billed = sum(r['input_tokens'] - r['cache_read'] for r in cached)
    saved = uncached_tok - cached_billed
    return {
        'model': model,
        'tokens_saved': saved,
        'savings_usd': round(saved / 1_000_000 * per_mtok, 6),
    }

for m in ['haiku', 'sonnet']:
    r = compute_savings_v2(MOCK_UNCACHED, MOCK_CACHED, model=m)
    print(f"{r['model']:7}: saved {r['tokens_saved']} tokens = ${r['savings_usd']}")

# Sonnet savings are 12× larger (ratio of input token prices 3.00/0.25 = 12×).
# This is why prompt caching has an even bigger ROI on Sonnet.

### Exercise 2 — Cache-Friendly Conversation Restructure

A naive multi-turn conversation repeats the full history in every call.
Restructure it so the **fixed system context** is cached and only the **new messages** change.

In [ ]:
# Exercise 2: Given this naive conversation structure, restructure it for optimal caching.
# The system prompt + static context should be in 'system' with cache_control.
# Only the user/assistant message history should change per call.
#
# naive_call = {
#   'model': 'claude-haiku-4-5-20251001',
#   'system': 'You are a Python expert. ' + LARGE_CONTEXT,
#   'messages': [{...history so far...}, {'role': 'user', 'content': new_question}]
# }
#
# TODO: rewrite as cache_friendly_call(...) that puts system prompt + context
# in the cacheable system block, keeping messages variable.


In [ ]:
# Exercise 2 — Answer Key
LARGE_CONTEXT = 'This system prompt is intentionally long. ' * 30

def cache_friendly_call(new_question: str, history: list, system: str, context: str) -> dict:
    # Put the large, fixed prefix in a cache_control block.
    # The user/assistant history in messages changes every turn — that's fine,
    # Anthropic only caches the SYSTEM prefix, not the messages array.
    return {
        'model': 'claude-haiku-4-5-20251001',
        'max_tokens': 256,
        'system': [
            {
                'type': 'text',
                'text': system + '\n\n' + context,
                'cache_control': {'type': 'ephemeral'}  # <- this is the only change
            }
        ],
        'messages': history + [{'role': 'user', 'content': new_question}],
    }

call = cache_friendly_call(
    new_question='What is the GIL?',
    history=[],
    system='You are a Python expert.',
    context=LARGE_CONTEXT,
)
print('system type:', type(call['system']).__name__)
print('cache_control:', call['system'][0]['cache_control'])
print('messages:', call['messages'])
# The messages list grows with each turn; the system block stays identical.
# This guarantees a cache hit on every follow-up question.

---
## What's Next

- **Anthropic prompt caching docs:** anthropic.com/docs/build-with-claude/prompt-caching — covers tool caching and multi-turn patterns
- **Cache for tool definitions:** pass large tool JSON in a cached system block when you have 10+ tools defined
- **Example 142 — Semantic Caching:** cache at the *question* level (similar questions reuse the same answer) instead of the prompt-prefix level
- **Example 143 — Context Compression:** reduce the system prompt size before caching to hit the 1024-token threshold more cheaply
- **Batch evaluation pattern:** use caching when running the same judge prompt against 100+ outputs — save 90% on input tokens